In [5]:
import pandas as pd

titanic = pd.read_csv("../data/titanic_propre.csv")

ports = pd.DataFrame({
    "embark_town": ["Southampton", "Cherbourg", "Queenstown"],
    "pays": ["Angleterre", "France", "Irlande"],
})
ports

,embark_town,pays
0,Southampton,Angleterre
1,Cherbourg,France
2,Queenstown,Irlande


In [6]:
nb_avant = len(titanic)
titanic_pays = titanic.merge(ports, on="embark_town", how="left", validate="many_to_one")
nb_apres = len(titanic_pays)

assert nb_apres == nb_avant
print(f"Lignes avant : {nb_avant}, après : {nb_apres}")
print(f"Pays manquants : {titanic_pays['pays'].isna().sum()}")

Lignes avant : 889, après : 889
Pays manquants : 0


In [7]:
print(titanic_pays["pays"].value_counts())
print()

bilan_par_pays = titanic_pays.groupby("pays").agg(
    passagers=("survived", "size"),
    taux_survie=("survived", "mean"),
).round(3)
bilan_par_pays

pays
Angleterre    644
France        168
Irlande        77
Name: count, dtype: int64



,passagers,taux_survie
pays,,
Angleterre,644,0.337
France,168,0.554
Irlande,77,0.390


In [8]:
ports_sans_irlande = ports[ports["embark_town"] != "Queenstown"]

jointure_inner = titanic.merge(
    ports_sans_irlande, on="embark_town", how="inner", validate="many_to_one"
)
jointure_left = titanic.merge(
    ports_sans_irlande, on="embark_town", how="left", validate="many_to_one"
)

print(f"inner : {len(jointure_inner)} lignes")
print(f"left : {len(jointure_left)} lignes, dont {jointure_left['pays'].isna().sum()} sans pays")
print(jointure_left["pays"].value_counts(dropna=False))

inner : 812 lignes
left : 889 lignes, dont 77 sans pays
pays
Angleterre    644
France        168
NaN            77
Name: count, dtype: int64


## Que se passe-t-il quand Queenstown manque dans `ports` ?

- **`how="inner"`** : on ne garde que les lignes dont le port existe dans les deux tables. Les 77 passagers de Queenstown disparaissent sans prévenir : 812 lignes au lieu de 889.
- **`how="left"`** : on garde toutes les lignes de `titanic`. Les 77 passagers de Queenstown restent, mais leur colonne `pays` est vide (`NaN`) : 889 lignes dont 77 sans pays.

Dans les deux cas, la jointure ne plante pas : l'erreur est silencieuse. Seul le contrôle du nombre de lignes (`assert`) et le comptage des valeurs manquantes permettent de la repérer. Avec `inner`, un `groupby("pays")` aurait donné des résultats faux sans aucun signe visible.